In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sentence_transformers import SentenceTransformer

In [2]:
import joblib

In [3]:
from sklearn.datasets import fetch_20newsgroups

news_grouptrain = fetch_20newsgroups(subset='train',shuffle=True,random_state=42,data_home='./dataset')

csv = pd.DataFrame({
    'text' : news_grouptrain.data,
    'Category' : news_grouptrain.target
})

print(csv.head())
print(len(csv.Category))
print(csv.shape)
print("\nCategories:", news_grouptrain.target_names)

                                                text  Category
0  From: lerxst@wam.umd.edu (where's my thing)\nS...         7
1  From: guykuo@carson.u.washington.edu (Guy Kuo)...         4
2  From: twillis@ec.ecn.purdue.edu (Thomas E Will...         4
3  From: jgreen@amber (Joe Green)\nSubject: Re: W...         1
4  From: jcm@head-cfa.harvard.edu (Jonathan McDow...        14
11314
(11314, 2)

Categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [8]:
model_name = "BAAI/bge-base-en-v1.5"
model = SentenceTransformer(model_name)
embedding_vector = joblib.load('/content/embeddings.joblib')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
print(len(embedding_vector))

11314


In [12]:
def preprocess(text):
  text = text.strip()

  return text

def cosine_similarity(v1,v2):
  if hasattr(v1,'Detach'):
    v1 = v1.detach().cpu().numpy()
  v1 = np.asarray(v1,dtype=np.float32).ravel()

  if hasattr(v2,'Detach'):
    v2 = v2.detach().cpu().numpy()
  v2 = np.asarray(v2,dtype=np.float32).ravel()

  if v2.ndim == 1:
    v2 = v2.ravel()
    denom = np.linalg.norm(v1) * np.linalg.norm(v2)
    return float(0.0 if denom == 0 else np.dot(v1,v2) / denom)

  v2 = np.atleast(v2)
  v1_norm = np.linalg.norm(v1)
  v2_norm = np.linalg.norm(v2,axis=1)
  denom = v1_norm * v2_norm
  with np.errstate(divide='ignore',invalid='ignore'):
    sims = (v2 @ v1)/np.where(denom == 0,1.0,denom)
  sims[denom == 0] = 0.0
  return sims.tolist()


def top_k_greatest_indices(lst,k):
    indexed_list = list(enumerate(lst))

    sorted_by_value = sorted(indexed_list, key=lambda x: x[1], reverse=True)

    top_k_indices = [index for index, value in sorted_by_value[:k]]
    return top_k_indices


In [28]:
def retrieval_document(query,embeddings,model,top_k = 5):
  query_clean = preprocess(query)
  query_embedding = model.encode(query_clean,convert_to_tensor=False).astype(np.float32)

  cosine_score = []

  for x in embeddings:

    if hasattr(x,'detach'):
      x = x.detach().cpu().numpy()
    x = np.asarray(x,dtype=np.float32)

    cosine_score.append(cosine_similarity(query_embedding,x))

    top_results = top_k_greatest_indices(cosine_score, k=top_k)

    print(f"Query: {query}")
    for idx in top_results:
        print(f"Document: {csv.iloc[idx]['text'][:200]}...")
        print(f"Category: {news_grouptrain.target_names[csv.iloc[idx]['Category']]}...")
        print("\n\n")




In [31]:
example_query = "Linear Algebra"
retrieval_document(example_query, embedding_vector, model, top_k = 5)

Streaming output truncated to the last 5000 lines.
Distribution: world
NNTP-Posting-Host: carson.u.washin...
Category: comp.graphics...



Document: From: msc_wdqn@jhunix.hcf.jhu.edu (Daniel Q Naiman)
Subject: Geometry package
Organization: Homewood Academic Computing, Johns Hopkins University, Baltimore, Md, USA
Lines: 11
Distribution: world
NNTP...
Category: comp.graphics...



Query: Linear Algebra
Document: From: talluri@osage.csc.ti.com (Raj Talluri)
Subject: Point of intersection of n lines
Keywords: robust statistics
Nntp-Posting-Host: osage
Organization: Texas Instruments
Lines: 21

Hi,

Can anybody ...
Category: comp.graphics...



Document: From: mkramer@world.std.com (Mark W Kramer)
Subject: Re: Seventh Century A.D. Armenian Math Problems
Keywords: philosphy, Greece, Persians, math 
Organization: The World Public Access UNIX, Brookline,...
Category: talk.politics.mideast...



Document: From: disser@engin.umich.edu (David Disser)
Subject: 2D bitmap interpolation
Organization